In [ ]:
# import os
# print(os.getcwd())  
    

In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from models.model import SimpleMLP
from models.utils import get_weight_matrices
from comparison.matrix_comparison import *


In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# Hyperparameters
input_size = 28 * 28  # MNIST images are 28x28
hidden_size = 32
output_size = 10  # 10 classes for digits 0-9
batch_size = 64
learning_rate = 0.001
num_epochs = 5

In [4]:
# Load dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)


In [5]:
# Initialize model, loss, and optimizer
model = SimpleMLP(input_size, hidden_size, output_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [6]:
# Training loop
print(f"Starting training using ${device}...")
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        # Reshape input
        data = data.view(data.shape[0], -1).to(device)
        targets = targets.to(device)

        # Forward pass
        scores = model(data)
        loss = criterion(scores, targets)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}")

print("Training complete!")

Starting training using $cuda...
Epoch [1/5], Step [0/938], Loss: 2.3224
Epoch [1/5], Step [100/938], Loss: 0.7237
Epoch [1/5], Step [200/938], Loss: 0.4355
Epoch [1/5], Step [300/938], Loss: 0.4945
Epoch [1/5], Step [400/938], Loss: 0.5260
Epoch [1/5], Step [500/938], Loss: 0.3998
Epoch [1/5], Step [600/938], Loss: 0.3281
Epoch [1/5], Step [700/938], Loss: 0.3297
Epoch [1/5], Step [800/938], Loss: 0.3018
Epoch [1/5], Step [900/938], Loss: 0.2698
Epoch [2/5], Step [0/938], Loss: 0.2454
Epoch [2/5], Step [100/938], Loss: 0.2680
Epoch [2/5], Step [200/938], Loss: 0.4013
Epoch [2/5], Step [300/938], Loss: 0.2175
Epoch [2/5], Step [400/938], Loss: 0.2895
Epoch [2/5], Step [500/938], Loss: 0.2578
Epoch [2/5], Step [600/938], Loss: 0.2720
Epoch [2/5], Step [700/938], Loss: 0.1720
Epoch [2/5], Step [800/938], Loss: 0.4431
Epoch [2/5], Step [900/938], Loss: 0.4761
Epoch [3/5], Step [0/938], Loss: 0.5105
Epoch [3/5], Step [100/938], Loss: 0.3456
Epoch [3/5], Step [200/938], Loss: 0.2956
Epoch [

In [7]:
# # Define a simple MLP
# mlp = nn.Sequential(
#     nn.Linear(3, 2),  # Input layer (size 3) -> Hidden layer (size 2)
#     nn.ReLU(),
#     nn.Linear(2, 2)   # Hidden layer (size 2) -> Output layer (size 2)
# )

# # Print the weights
# for i, layer in enumerate(mlp):
#     if isinstance(layer, nn.Linear):
#         print(f"Layer {i}:\nWeights\n{layer.weight.data}\nBias\n{layer.bias.data}\n")

# Print the model's structure
print(model)

# Print the weights
for i, layer in enumerate(model.model):
    if isinstance(layer, nn.Linear):
        print(f"Layer {i}:\nWeights\n{layer.weight.data}\nBias\n{layer.bias.data}\n")

SimpleMLP(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=10, bias=True)
  )
)
Layer 0:
Weights
tensor([[-0.0087, -0.0313,  0.0093,  ..., -0.0317,  0.0073,  0.0061],
        [-0.0130, -0.0442, -0.0158,  ..., -0.0617, -0.0639, -0.0183],
        [ 0.0381,  0.0344, -0.0146,  ...,  0.0213, -0.0122,  0.0238],
        ...,
        [ 0.0038,  0.0163,  0.0108,  ..., -0.0216, -0.0015, -0.0112],
        [-0.0222, -0.0102,  0.0299,  ..., -0.0218,  0.0328, -0.0303],
        [-0.0139, -0.0196,  0.0172,  ...,  0.0002,  0.0389, -0.0197]],
       device='cuda:0')
Bias
tensor([-0.0100,  0.0634, -0.0204, -0.0287,  0.0341, -0.0142,  0.0214,  0.0220,
        -0.0244, -0.0035,  0.0128, -0.0030,  0.0275,  0.0390,  0.0209,  0.0080,
        -0.0125,  0.0004,  0.0244,  0.0113, -0.0040,  0.0468, -0.0157,  0.0103,
        -0.0428, -0.0272,  0.0059, -0.0108,  0.0320,  0.0140,  0.0047, -0.0342],
       device='cuda:0'

In [34]:
weights = get_weight_matrices(model)
print(len(weights))  # 2
print(f"First weight: {weights[0].shape}")  # (32, 784)
print(f"Second weight: {weights[1].shape}")  # (10, 32)

2
First weight: (32, 784)
Second weight: (10, 32)


In [35]:
untrained_model = model = SimpleMLP(input_size, hidden_size, output_size).to(device)
untraine_weights = get_weight_matrices(untrained_model)
print(len(untraine_weights))  # 2
print(f"First weight: {untraine_weights[0].shape}")  # (32, 784)
print(f"Second weight: {untraine_weights[1].shape}")  # (10, 32)

2
First weight: (32, 784)
Second weight: (10, 32)


In [38]:
nuclear_norm = matrix_difference_nuclear(weights[1], untraine_weights[1])
print(f"Nuclear norm difference: {nuclear_norm}")

Nuclear norm difference: 10.628751754760742
